In [ ]:
import numpy as np
import ants
import matplotlib.pyplot as plt
import os
import gc
import nibabel as nib
from mpl_toolkits.axes_grid1 import ImageGrid
import logger
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.utils import to_categorical
from antspynet import brain_extraction


import matplotlib.pyplot as plt
import gc
import seaborn as sns

from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from PIL import Image
import tempfile

In [ ]:
def winsorize_image(image_data, lower_percentile=0, upper_percentile=99.9): #reduz valores extremos
    lower_bound = np.percentile(image_data, lower_percentile)
    upper_bound = np.percentile(image_data, upper_percentile)
    winsorized_data = np.clip(image_data, lower_bound, upper_bound)
    return winsorized_data

def normalize_image_min(image_data): #normalizar 
    min_val = np.min(image_data)
    max_val = np.max(image_data)
    normalized_data = (image_data - min_val) / (max_val - min_val)
    return normalized_data

# Função para processar uma única imagem
def process_image(img_path, template, registro='Affine', orientation='false'):
    try:
        logger.info(f"Inicio processamento: {img_path}")
        # Carrega a imagem
        image = ants.image_read(img_path,reorient=orientation)

        # Registra pra padronizar shape da imagem
        registration = ants.registration(fixed=template, moving=image, type_of_transform=registro)
        affine_image = registration['warpedmovout']
        brain_masked = affine_image

        # Cria template pra máscara
        prob_mask = brain_extraction(affine_image, modality='t1')
        # logger.info(f"Template obtido.")

        # # Cria a máscara
        mask = ants.get_mask(prob_mask, low_thresh=0.5)
        # logger.info(f"Máscara aplicada.")

        # # Máscara do cérebro e extração
        brain_masked = ants.mask_image(affine_image, mask)
        # logger.info(f"Extração.")

        # Bias Field Correction
        #image = ants.from_numpy(data, origin=image.origin, spacing=image.spacing, direction=image.direction)
        image = ants.n4_bias_field_correction(brain_masked, shrink_factor=2)
        data = image.numpy()
        logger.info(f"Bias Corrigido.")

        # Winsorizing
        data = winsorize_image(data, 0, 99.9)
        logger.info(f"Winsorized.")

        # Normalização
        data = normalize_image_min(data)
        image = ants.from_numpy(data, origin=brain_masked.origin, spacing=brain_masked.spacing, direction=brain_masked.direction)

        logger.info(f"Imagem {img_path} processada.")

        #ants.image_write(image, output_path)
        #logger.info(f"Imagem salva: {os.path.basename(output_path)}")

        gc.collect()

        return image
        
    except Exception as e:
        logger.error(f"Erro ao processar a imagem {img_path}: {e}")
        return None
    
def plot_views(image, k=0):
    fig, axs = plt.subplots(1, 3)
    size = image.shape

    axs[0].imshow(np.rot90(image[size[0]//2, :, :], k=k), cmap='grey')
    axs[0].set_title("esperado: sagital")
    axs[0].axis('off')
    
    axs[1].imshow(np.rot90(image[:, size[1]//2, :]), cmap='grey')
    axs[1].set_title("esperado: coronal")
    axs[1].axis('off')

    axs[2].imshow(np.rot90(image[:, :, size[2]//2], k=k), cmap='grey')
    axs[2].set_title("esperado: axial")
    axs[2].axis('off')
    
    fig.tight_layout(rect=[0, 0, 1, 0.8])
    plt.show()

def plot_one_view(image, index=80, rot=0, title=''):
    image = np.rot90(image[:, :, index], k=rot)
    plt.figure(figsize=(10,6))
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

def unificar_tamanhos_com_padding(lista_de_imagens):
    max_altura = 0
    max_largura = 0
    for img in lista_de_imagens:
        altura, largura = img.shape
        if altura > max_altura:
            max_altura = altura
        if largura > max_largura:
            max_largura = largura

    imagens_uniformes = []
    for img in lista_de_imagens:
        fundo = np.zeros((max_altura, max_largura))
        
        altura_img, largura_img = img.shape
        y_offset = (max_altura - altura_img) // 2
        x_offset = (max_largura - largura_img) // 2
        
        fundo[y_offset:y_offset+altura_img, x_offset:x_offset+largura_img] = img
        imagens_uniformes.append(fundo)
        
    return imagens_uniformes

def plot_views_uniforme_final(image, k=0, sag_idx=90, cor_idx=110, ax_idx=110, figsize=(15, 5), axes_pad=0.3):
    fig = plt.figure(figsize=figsize)
    grid = ImageGrid(fig, 111,
                    nrows_ncols=(1, 3),
                    axes_pad=axes_pad)
    
    slices_originais = [
        np.rot90(image[sag_idx, :, :], k=k),
        np.rot90(image[:, cor_idx, :], k=k),
        np.rot90(image[:, :, ax_idx], k=k)
    ]
    
    slices_uniformizadas = unificar_tamanhos_com_padding(slices_originais)
    
    titles = ["Sagital", "Coronal", "Axial"]

    for ax, im_slice, title in zip(grid, slices_uniformizadas, titles):
        ax.imshow(im_slice, cmap='gray')
        ax.set_title(title)
        ax.axis('off')

    plt.show()

# realizar predições e armazenar em um vetor
def get_predictions(images, labels, batch_size, best_model):
    pred = []

    for i in range(0, len(images), batch_size):
        final = min(i + batch_size, len(images))
        
        # Fazendo predição para o lote atual
        batch_pred = best_model.predict(images[i:final])
        pred.append(batch_pred)

    # Concatenando as predições e os rótulos verdadeiros
    pred = np.concatenate(pred)

    # Convertendo as predições para rótulos (a classe com maior probabilidade)
    true_labels = np.argmax(labels, axis=1)
    pred_labels = np.argmax(pred, axis=1)
    return pred_labels, true_labels, pred

def plot_confusion_matrix(y_true, y_pred, dir, subset, class_names):
    # Calcular a matriz de confusão
    cm = confusion_matrix(y_true, y_pred)

    # Plotando a matriz de confusão
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names,  annot_kws={"size": 14})
    plt.xlabel('Previsões')
    plt.ylabel('Valores Reais')
    plt.title('Matriz de Confusão')
    plt.savefig(f'{dir}/{subset}_confusion_matrix.png')
    plt.show()

def plot_custom_confusion_matrix(cm, y_labels, x_labels, dir, subset):
    plt.figure(figsize=(10, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=x_labels, yticklabels=y_labels, 
                annot_kws={"size": 14})
    plt.xlabel('Previsões')
    plt.ylabel('Valores Reais')
    plt.title('Matriz de Confusão Ajustada')
    plt.savefig(f'{dir}/{subset}_custom_confusion_matrix.png', bbox_inches='tight')
    plt.show()

def get_classification_report(y_true, y_pred, dir, subset):
    # Gerar relatório
    report = classification_report(y_true, y_pred)
    print(report)

    # Escrevendo o relatório em um arquivo .txt
    with open(os.path.join(dir, f"{subset}_classification_report.txt"), "w") as file:
        file.write(report)

def load_nifti_data_balanced(base_dir, class_names, target=1000):
    images = []
    labels = []
    paths = []
    
    # Caminhos das subpastas
    for label in class_names:
        print(f"carregando diretório {label}")
        label_dir = os.path.join(base_dir, label)
        count = 0

        names = os.listdir(label_dir)
        #np.random.shuffle(names)
        for fname in names:
            #if count < target:
            img_path = os.path.join(label_dir, fname)
            img = nib.load(img_path).get_fdata(dtype=np.float16)
            paths.append(img_path)
            images.append(img)
            labels.append(label)
            count += 1

        print(f"diretório carregado {count}")

    # Codificando os rótulos
    label_encoder = LabelEncoder()
    label_encoder.classes_ = np.array(class_names)
    labels_encoded = label_encoder.transform(labels)
    labels_one_hot = to_categorical(labels_encoded, num_classes=len(class_names))

    # Convertendo para arrays NumPy
    images = np.array(images).reshape((-1, *images[0].shape, 1))
    labels_one_hot = np.array(labels_one_hot)
    
    # Embaralhar os dados
    images, labels_one_hot, paths = shuffle(images, labels_one_hot, paths, random_state=42)
    
    return images, labels_one_hot, paths, label_encoder.classes_


# Função para criar o PDF
def create_pdf(y_paths, y_images, y_true_labels, y_pred_labels, y_pred, output_pdf_path, class_names):
    c = canvas.Canvas(output_pdf_path, pagesize=letter)
    width, height = letter  # Dimensões da página no PDF

    #for image, name in zip(y_images, y_paths):
    for i in range(0, len(y_images)):
        true = ''
        pred = ''
        # Carregar a imagem NIfTI e obter a fatia 2D no eixo Z
        # nifti_image = load_nifti_image_pdf(item)
        nifti_image = y_images[i][:, :, 88, 0]

        # Converter a fatia 2D para uma imagem 8-bit (grayscale) para visualização
        img = Image.fromarray(np.uint8(nifti_image / np.max(nifti_image) * 255))  # Normalizar e converter
        img = img.convert("RGB")  # Garantir que a imagem tenha 3 canais (RGB)

        # Redimensionar a imagem para se ajustar ao tamanho da página
        img_width, img_height = img.size
        aspect_ratio = img_height / float(img_width)
        new_width = width * 0.2  # Definir largura como 80% da largura da página
        new_height = new_width * aspect_ratio
        img = img.resize((int(new_width), int(new_height)))

        # Criar um arquivo temporário para salvar a imagem
        with tempfile.NamedTemporaryFile(delete=False, suffix=".png") as temp_file:
            temp_file_path = temp_file.name
            img.save(temp_file_path)

        # configurar para printar as 7 fatias em uma página inteira, com as informações de label predito e esperado

        # Colocar a imagem no PDF usando o caminho temporário
        if i % 12 < 4:
            x = 80
        elif i % 12 < 8:
            x = width - 2.35*new_width - 80
        else:
            x = width - new_width - 80

        y = height - (new_height + 80)*((i%4)+1)

        c.drawImage(temp_file_path, x, y, width=new_width, height=new_height)

        # Escrever os rótulos
        true_label = y_true_labels[i]
        pred_label = y_pred_labels[i]

        #ver como transformar os labels de maneira inteligente
        true = class_names[true_label]
        pred = class_names[pred_label]

        # Definir a cor para os rótulos
        if true_label == pred_label:
            pred_color = (0, 1, 0)  # Verde
        else:
            pred_color = (1, 0, 0)  # Vermelho
        
        #Nome paciente (em preto)
        c.setFont("Helvetica", 12)
        c.setFillColorRGB(0, 0, 0)  # Preto
        c.drawString(x+24, y+new_height+50, f"{os.path.basename(y_paths[i])}")

        # Rótulo esperado (em preto)
        c.setFont("Helvetica", 12)
        c.setFillColorRGB(0, 0, 0)  # Preto
        c.drawString(x+26, y+new_height+35, f"Expected: {true}")

        # Rótulo predito
        c.setFont("Helvetica", 12)
        c.setFillColorRGB(*pred_color)  # Verde ou Vermelho
        c.drawString(x+26, y+new_height+20, f"Predicted: {pred}")

        # Rótulo predito
        c.setFont("Helvetica", 12)
        c.setFillColorRGB(*pred_color)  # Verde ou Vermelho
        c.drawString(x+26, y+new_height+5, f"Prob: {max(y_pred[i])*100:.2f}%")

        # Avançar para a próxima imagem
        i += 1
        
        # Adicionar uma nova página no PDF a cada 2 imagens (se necessário)
        if i % 12 == 0:  # Por exemplo, a cada 2 imagens, adicionamos uma nova página
            c.showPage()

    # Salvar o PDF
    c.save()

In [ ]:
dir = "/mnt/c/Users/Bruno/Documents/Github"

oasis_dir = f"/mnt/c/Users/Bruno/Desktop/IANS/OAS2_RAW_PART2"
output_dir = f"/mnt/c/Users/Bruno/Desktop/IANS/OASIS_2_RAW"
os.makedirs(output_dir, exist_ok=True)

oasis_data_path = "/mnt/c/Users/Bruno/Desktop/IANS/oasis_2_data.xlsx"
oasis_data = pd.read_excel(oasis_data_path)

template_path = f"{dir}/Alzheimer-CNN-Detection/pre_processing/mni_icbm152_nlin_asym_09c_nifti/mni_icbm152_nlin_asym_09c/mni_icbm152_t1_tal_nlin_asym_09c.nii"
template = ants.image_read(template_path)
mask_path = f"{dir}/Alzheimer-CNN-Detection/pre_processing/mni_icbm152_nlin_asym_09c_nifti/mni_icbm152_nlin_asym_09c/mni_icbm152_t1_tal_nlin_asym_09c_mask.nii"
mask = ants.image_read(mask_path)

SLICE_NII_IDX0 = slice(15, 175)
SLICE_NII_IDX1 = slice(21, 210)
SLICE_NII_IDX2 = slice(7, 164)

In [ ]:
groups = oasis_data[['Subject ID', 'CDR']]
groups = groups.set_index('Subject ID', drop=False)
print(groups)

In [ ]:
subjects = os.listdir(oasis_dir)

print(subjects)

for sub in subjects:
    if sub.endswith("MR1"):
        group = groups.loc[sub.split('_MR')[0], 'CDR'].iloc[0]
        sub_path = f"{oasis_dir}/{sub}/RAW/mpr-1.nifti.img"
        print(sub)
        img = ants.image_read(sub_path)
        sub_output_dir = f"{output_dir}/{group}"
        os.makedirs(sub_output_dir, exist_ok=True)
        ants.image_write(img, f"{sub_output_dir}/{sub}.nii.gz")

In [ ]:
img = ants.image_read(f"{test_dir}/0.0/{os.listdir(f"{test_dir}/0.0")[0]}")

data = img.numpy()[SLICE_NII_IDX0, SLICE_NII_IDX1, SLICE_NII_IDX2]
print(data.shape)
plot_views_uniforme_final(data, 1, 90, 40, 82)

In [ ]:
# PLOTS ADNI
adni_proc_path = f"/mnt/c/Users/Bruno/Desktop/IANS/Alzheimer/test_affine/cn"

adni_img_proc = ants.image_read(f"{adni_proc_path}/{os.listdir(adni_proc_path)[6]}")

adni_wins = winsorize_image(adni_img_proc.numpy(), 0, 99.9)

plot_views_uniforme_final(adni_wins, 0, 90, 120, 85)
plot_views_uniforme_final(adni_img_proc.numpy(), 0, 90, 120, 85)

In [ ]:
# PLOTS OASIS
raw_path = f"{output_dir}/0.0"
proc_path = f"{test_dir}/0.0"

oasis_img_raw = ants.image_read(f"{raw_path}/{os.listdir(raw_path)[0]}")
oasis_img_proc = ants.image_read(f"{proc_path}/{os.listdir(proc_path)[0]}")

plot_views_uniforme_final(oasis_img_raw.numpy(), 0, 90, 130, 82)

plot_views_uniforme_final(oasis_img_proc.numpy(), 0, 90, 120, 82)

In [ ]:
adni_classes = ['CN', 'EMCI', 'MCI', 'LMCI', 'AD']
oasis_classes = ['0.0', '0.5', '1.0']
class_names = oasis_classes

model = load_model(f"/mnt/c/Users/Bruno/Desktop/IANS/OASIS_2_PROCESSED/binary_classifier_150_epochs_batch_64_5_classes.keras")

model.summary()

test_dir = f"/mnt/c/Users/Bruno/Desktop/IANS/OASIS_2_PROCESSED"
results_dir = f"{test_dir}/results"
os.makedirs(results_dir, exist_ok=True)

In [ ]:
test_images, test_labels, test_paths, _ = load_nifti_data_balanced(test_dir, class_names)

In [ ]:
test_images_crop = []

for i in range(len(test_images)):
    backup = test_images[i]
    # print(backup.shape)
    test_images_crop.append(backup[SLICE_NII_IDX0, SLICE_NII_IDX1, SLICE_NII_IDX2])

test_images_crop = np.array(test_images_crop)

print(test_images_crop.shape)

In [ ]:
# Realizar predições para dados do conjunto validação
test_pred_labels, test_true_labels, test_pred = get_predictions(test_images_crop, test_labels, 8, model)

In [ ]:
n = len(os.listdir(results_dir))

# if (n > 0):
#     if (len(os.listdir(os.path.join(results_dir, f'test_{n}'))) < 3): 
#         for item in os.listdir(os.path.join(results_dir, f"test_{n}")):
#             os.remove(os.path.join(results_dir,  f"test_{n}", item))
#         os.removedirs(os.path.join(results_dir, f'test_{n}'))
#         n -= 1

folder_name = f"test_{str(n+1)}"
folder_dir = os.path.join(results_dir, folder_name)
os.makedirs(folder_dir, exist_ok=True)
print(f"pasta {folder_name} criada")

In [ ]:
# Obter métricas da valiadação e salvá-las em um arquivo
# get_classification_report(test_true_labels, test_pred_labels, results_dir, 'test')

# Obter matriz de confusão
plot_confusion_matrix(test_true_labels, test_pred_labels, folder_dir, 'test_5x5', class_names)

In [ ]:
# all_possible_numeric_labels = [0.0, 0.5, 1.0, 2.0, 3.0]

cm_adjusted = confusion_matrix(test_true_labels, test_pred_labels)[0:3, :]

# Chame a nova função para plotar a matriz ajustada (3x5)
plot_custom_confusion_matrix(cm_adjusted, oasis_classes, adni_classes, folder_dir, 'test_3x5')


In [ ]:
gathered_test_pred = []

for i in range(len(test_pred_labels)):
    if test_pred_labels[i] == 0:
        gathered_test_pred.append(0)
    elif test_pred_labels[i] < 4 and test_pred_labels[i] > 0:
        gathered_test_pred.append(1)
    else:
        gathered_test_pred.append(2)

# all_possible_numeric_labels = [0.0, 0.5, 1.0, 2.0, 3.0]

cm_3x3_adjusted = confusion_matrix(test_true_labels, gathered_test_pred)[0:3, :]

# Chame a nova função para plotar a matriz ajustada (3x5)
plot_custom_confusion_matrix(cm_3x3_adjusted, oasis_classes, oasis_classes, folder_dir, 'test_3x3')


In [ ]:
false_cn_name = []
false_cn_index = []

for i in range(len(test_images)):
    if test_pred_labels[i] != test_true_labels[i] and test_true_labels[i] == 0:
        false_cn_name.append(os.path.basename(test_paths[i]))
        false_cn_index.append(i)

In [ ]:
print(false_cn_name)

print(false_cn_index)

In [ ]:
def gerar_pdf_plots_volumetricos(imagens, indices, labels, nome_arquivo_pdf, incremento=5, n_slices=10):
    """
    Gera um PDF com uma página de capa para cada paciente, seguida por
    5 plots tri-planares (X, Y, Z) para cada imagem especificada pelos índices.
    
    A cada plot, o índice da fatia (slice) é incrementado.

    Parâmetros:
    - imagens (np.ndarray): Array 4D ou 5D (N, H, W, D, [C]) contendo todas as imagens.
    - indices (list ou np.ndarray): Lista de índices das imagens a serem plotadas.
    - labels (np.ndarray): Array de labels (provavelmente one-hot) para as imagens.
    - nome_arquivo_pdf (str): O nome do arquivo PDF a ser criado.
    - incremento (int): O valor a ser somado aos índices dos eixos a cada etapa.
    """
    
    print(f"Iniciando a geração do PDF: {nome_arquivo_pdf}")
    
    # Abre o arquivo PDF
    with PdfPages(nome_arquivo_pdf) as pdf:
        
        # 1. Loop sobre os índices das imagens que queremos plotar
        for idx in indices:
            print(f"Processando imagem de índice {idx}...")
            
            # Pega a imagem 3D específica
            try:
                imagem_3d = imagens[idx] # Esta agora tem shape (H, W, D, 1)
                shape = imagem_3d.shape
            except IndexError:
                print(f"  AVISO: Índice {idx} está fora dos limites. Pulando.")
                continue
            except Exception as e:
                print(f"  ERRO: Não foi possível acessar a imagem {idx}. Erro: {e}. Pulando.")
                continue

            # --- INÍCIO DA PÁGINA DE CAPA ---
            try:
                # Pega o nome da label. 
                # !!! IMPORTANTE: 'adni_classes' deve estar disponível no escopo 
                #     onde esta função é chamada (ex: variável global ou de célula).
                label_texto = adni_classes[labels[idx]]
                
                # Cria uma figura em branco para a capa
                fig_capa = plt.figure(figsize=(18, 6)) # Mesmo tamanho dos plots
                
                # Desliga os eixos
                plt.axis('off')
                
                # Adiciona o texto no centro
                texto_capa = f"Paciente: {idx}\n\nLabel: {label_texto}"
                
                fig_capa.text(0.5, 0.5, texto_capa, 
                              ha='center', va='center', 
                              fontsize=24, wrap=True)
                
                # Salva a figura da capa como uma página no PDF
                pdf.savefig(fig_capa)
                
                # Fecha a figura da capa para liberar memória
                plt.close(fig_capa)
                
            except NameError:
                print("  AVISO: A variável 'adni_classes' não foi encontrada.")
                print("         Não foi possível gerar a página de capa ou os títulos.")
                # Se 'adni_classes' não existir, pulamos a capa e os plots
                # (pois os plots também dependem dela para o suptitle)
                continue 
            except Exception as e:
                print(f"  AVISO: Não foi possível gerar a página de capa para {idx}. Erro: {e}")
                # Continua mesmo se a capa falhar
            # --- FIM DA PÁGINA DE CAPA ---


            # Define os slices (fatias) centrais como ponto de partida
            base_x = shape[0] // 2 - 45
            base_y = shape[1] // 2 - 45
            base_z = shape[2] // 2 - 45
            
            # 2. Loop para criar os 5 plots com incremento
            for i in range(n_slices):
                offset = i * incremento
                
                # Calcula os novos índices dos slices, com verificação de limites
                slice_x = min(base_x + offset, shape[0] - 1)
                slice_y = min(base_y + offset, shape[1] - 1)
                slice_z = min(base_z + offset, shape[2] - 1)
                
                # 3. Cria o "bloco" do matplotlib (figura com 3 subplots)
                fig, axes = plt.subplots(1, 3, figsize=(18, 6))
                
                # --- Plot 1: Vista X (Sagital) ---
                vista_x = imagem_3d[slice_x, :, :, 0]
                axes[0].imshow(vista_x.T, cmap='gray', origin='lower')
                axes[0].set_title(f'Vista X (Sagital) - Slice: {slice_x}')
                axes[0].set_xlabel('Eixo Y')
                axes[0].set_ylabel('Eixo Z')

                # --- Plot 2: Vista Y (Coronal) ---
                vista_y = imagem_3d[:, slice_y, :, 0]
                axes[1].imshow(vista_y.T, cmap='gray', origin='lower')
                axes[1].set_title(f'Vista Y (Coronal) - Slice: {slice_y}')
                axes[1].set_xlabel('Eixo X')
                axes[1].set_ylabel('Eixo Z')

                # --- Plot 3: Vista Z (Axial/Transversal) ---
                vista_z = imagem_3d[:, :, slice_z, 0]
                axes[2].imshow(vista_z.T, cmap='gray', origin='lower')
                axes[2].set_title(f'Vista Z (Axial) - Slice: {slice_z}')
                axes[2].set_xlabel('Eixo X')
                axes[2].set_ylabel('Eixo Y')

                # Título geral para a figura (a página do PDF)
                # (Esta linha assume que 'adni_classes' e 'np' estão disponíveis)
                label_texto_titulo = adni_classes[labels[idx]]
                fig.suptitle(f'Imagem {idx} - Offset: +{offset} - label {label_texto_titulo}', fontsize=16)
                plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Ajusta para o super-título
                
                # Salva a figura atual como uma nova página no PDF
                pdf.savefig(fig)
                
                # Fecha a figura para liberar memória
                plt.close(fig)
    
    print(f"Geração do PDF '{nome_arquivo_pdf}' concluída.")

In [ ]:
num_imagens = len(false_cn_index)
dim_h, dim_w, dim_d = (test_images[0].shape[0], test_images[0].shape[1], test_images[0].shape[2])

# Cria um gradiente simples para ser fácil de visualizar
x, y, z = np.indices((dim_h, dim_w, dim_d))
dados_base = (x + y + z) / (dim_h + dim_w + dim_d) # Normalizado entre 0 e ~1

# Cria o vetor de imagens (4D)
vetor_de_imagens = test_images
 
print(f"Forma do vetor de imagens: {vetor_de_imagens.shape}")

# 3. NOME DO ARQUIVO DE SAÍDA
os.makedirs("pdf", exist_ok=True)
nome_do_arquivo = "pdf/falsos_cognitively_normal.pdf"

# 4. CHAMADA DA FUNÇÃO
gerar_pdf_plots_volumetricos(
    imagens=vetor_de_imagens, 
    indices=false_cn_index,
    labels=test_pred_labels,
    nome_arquivo_pdf=nome_do_arquivo,
    incremento=5
)

In [ ]:
template_images, template_labels, template_paths, _ = load_nifti_data_balanced(f"/mnt/c/Users/Bruno/Documents/GitHub/Alzheimer-CNN-Detection/pre_processing/mni_icbm152_nlin_asym_09c_nifti", ['mni_icbm152_nlin_asym_09c'])

In [ ]:
gerar_pdf_plots_volumetricos(
    imagens=template_images, 
    indices=[i for i in range(len(template_images))],
    labels=[0 for i in range(len(template_images))],
    nome_arquivo_pdf='pdf/templates.pdf',
    incremento=1,
    n_slices=50
)

In [ ]:
# # Criar pdf com predições
# test_pdf_path = os.path.join(results_dir, "oasis_test_predictions.pdf")
# create_pdf(test_paths, test_images, test_true_labels, test_pred_labels, test_pred, test_pdf_path, class_names)